# Validação - Senado/Brasil

Além da presença dos arquivos, verifica:
- senadores e períodos históricos por UF;
- despesas sem correspondência no mapa histórico;
- erros de coleta de matérias;
- recorte estadual dos votos.

Se ainda houver despesa sem senador histórico identificado, o status fica `revisar`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re

import pandas as pd

ROOT = Path("/Volumes/workspace/pi_ii_bronze/senado")
ANOS = [2023, 2024, 2025, 2026]

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

CONTAGEM_COMPLETA = False

def pasta_uf(uf):
    return ROOT / uf.lower()

def normalizar(valor):
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())

print("Raiz:", ROOT)


In [ ]:
# arquivos esperados
nomes = [
    "senadores.csv",
    "senadores_exercicios.csv",
]

for ano in ANOS:
    nomes += [
        f"materias_autorias_{ano}.csv",
        f"materias_{ano}.csv",
        f"despesas_{ano}.csv",
        f"votos_{ano}.csv",
        f"votacoes_{ano}.csv",
    ]

linhas = []

for uf in UFS:
    for nome in nomes:
        path = pasta_uf(uf) / nome

        linhas.append({
            "uf": uf,
            "arquivo": nome,
            "existe": path.exists(),
            "tamanho_mb": (
                round(path.stat().st_size / 1024**2, 2)
                if path.exists() else None
            ),
        })

arquivos = pd.DataFrame(linhas)
faltantes = arquivos.loc[
    ~arquivos["existe"]
].copy()

print("Arquivos esperados:", len(arquivos))
print("Faltantes:", len(faltantes))

if len(faltantes):
    display(faltantes)


In [ ]:
# senadores históricos por UF
resumo_senadores = []

for uf in UFS:
    path = pasta_uf(uf) / "senadores.csv"
    path_ex = pasta_uf(uf) / "senadores_exercicios.csv"

    if not path.exists() or not path_ex.exists():
        continue

    df = pd.read_csv(
        path,
        sep=";",
        dtype=str,
        keep_default_na=False,
    )

    ex = pd.read_csv(
        path_ex,
        sep=";",
        dtype=str,
        keep_default_na=False,
    )

    resumo_senadores.append({
        "uf": uf,
        "senadores_historicos_unicos": len(df),
        "periodos_de_exercicio": len(ex),
    })

senadores_df = pd.DataFrame(resumo_senadores)

senadores_df.to_csv(
    ROOT / "_senadores_por_uf.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

display(senadores_df)


In [ ]:
# checagens rápidas do recorte
alertas = []

for uf in UFS:
    for ano in ANOS:
        # votos
        path = pasta_uf(uf) / f"votos_{ano}.csv"

        if path.exists() and path.stat().st_size > 0:
            amostra = pd.read_csv(
                path,
                sep=";",
                dtype=str,
                nrows=3000,
                keep_default_na=False,
            )

            candidatos = [
                c for c in amostra.columns
                if "uf" in normalizar(c)
            ]

            candidatos_parlamentar = [
                c for c in candidatos
                if "parlamentar" in normalizar(c)
                or normalizar(c) in {"siglauf", "ufparlamentar"}
            ]

            if candidatos_parlamentar:
                col = candidatos_parlamentar[0]

                valores = set(
                    amostra[col]
                    .astype(str)
                    .str.strip()
                    .str.upper()
                )
                valores.discard("")

                if not valores.issubset({uf}):
                    alertas.append(
                        f"{uf} / votos_{ano}: UF inesperada em {col}: "
                        f"{sorted(valores)}"
                    )

print("Alertas:", len(alertas))

for a in alertas[:100]:
    print(" -", a)


In [ ]:
def contar(path):
    total = 0

    for chunk in pd.read_csv(
        path,
        sep=";",
        dtype=str,
        chunksize=200_000,
        keep_default_na=False,
    ):
        total += len(chunk)

    return total

if CONTAGEM_COMPLETA:
    contagens = []

    for uf in UFS:
        for nome in nomes:
            path = pasta_uf(uf) / nome

            contagens.append({
                "uf": uf,
                "arquivo": nome,
                "registros": (
                    contar(path)
                    if path.exists()
                    else None
                ),
            })

    contagens_df = pd.DataFrame(contagens)

    contagens_df.to_csv(
        ROOT / "_contagens_brasil.csv",
        sep=";",
        index=False,
        encoding="utf-8",
    )

    display(contagens_df)
else:
    print("Contagem completa desativada.")

# valida os períodos históricos
mapa_exercicios = ROOT / "_senadores_exercicios_brasil.csv"
periodos_invertidos = 0

if mapa_exercicios.exists():
    ex = pd.read_csv(
        mapa_exercicios,
        sep=";",
        dtype=str,
        keep_default_na=False,
    )

    ini = pd.to_datetime(
        ex["data_inicio_exercicio"],
        errors="coerce",
    )
    fim = pd.to_datetime(
        ex["data_fim_exercicio"],
        errors="coerce",
    )

    mask = (
        ini.notna()
        & fim.notna()
        & (ini > fim)
    )

    periodos_invertidos = int(mask.sum())

    if periodos_invertidos:
        alertas.append(
            f"Senadores: {periodos_invertidos} períodos de exercício "
            "com data inicial posterior à final."
        )

# despesas sem mapa histórico
despesas_sem_mapa = []

for ano in ANOS:
    path = ROOT / f"_despesas_nao_distribuidas_{ano}.csv"

    if path.exists():
        n = contar(path)
    else:
        n = None

    despesas_sem_mapa.append({
        "ano": ano,
        "registros_nao_distribuidos": n,
    })

despesas_sem_mapa_df = pd.DataFrame(
    despesas_sem_mapa
)

display(despesas_sem_mapa_df)

total_sem_mapa = int(
    despesas_sem_mapa_df[
        "registros_nao_distribuidos"
    ]
    .fillna(0)
    .sum()
)

if total_sem_mapa > 0:
    alertas.append(
        f"CEAPS: {total_sem_mapa} registros ainda sem "
        "correspondência em senador com exercício na 57ª legislatura."
    )

# erros de matérias
auditoria_materias_path = ROOT / "_auditoria_materias.json"

erros_materias = {}

if auditoria_materias_path.exists():
    with auditoria_materias_path.open(
        "r", encoding="utf-8"
    ) as f:
        auditoria_materias = json.load(f)

    erros_materias = auditoria_materias.get(
        "erros",
        {},
    )

if erros_materias:
    alertas.append(
        f"Matérias: {len(erros_materias)} senadores "
        "com erro de consulta."
    )

print("Alertas finais:", len(alertas))

for a in alertas:
    print(" -", a)


In [ ]:
# status final
resumo_estados = (
    arquivos
    .groupby("uf", as_index=False)
    .agg(
        arquivos_esperados=("arquivo", "count"),
        arquivos_presentes=("existe", "sum"),
        tamanho_total_mb=("tamanho_mb", "sum"),
    )
)

resumo_estados["completo"] = (
    resumo_estados["arquivos_esperados"]
    == resumo_estados["arquivos_presentes"]
)

resumo_estados = resumo_estados.merge(
    senadores_df,
    on="uf",
    how="left",
)

resumo_estados.to_csv(
    ROOT / "resumo_brasil.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

ano_atual = datetime.now(timezone.utc).year
anos_parciais = [
    ano_atual
    if ano_atual in ANOS
    else None
]
anos_parciais = [
    x for x in anos_parciais
    if x is not None
]

status = {
    "ufs": len(UFS),
    "anos": ANOS,
    "anos_parciais": anos_parciais,
    "arquivos_esperados": int(len(arquivos)),
    "arquivos_faltantes": int(len(faltantes)),
    "periodos_exercicio_invertidos": periodos_invertidos,
    "despesas_sem_mapa_historico": total_sem_mapa,
    "erros_materias": len(erros_materias),
    "alertas": alertas,
    "status": (
        "ok"
        if len(faltantes) == 0 and len(alertas) == 0
        else "revisar"
    ),
    "validado_em_utc": datetime.now(timezone.utc).isoformat(),
}

with (ROOT / "_status_pipeline_brasil.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(
        status,
        f,
        ensure_ascii=False,
        indent=2,
    )

display(resumo_estados)
print(json.dumps(status, ensure_ascii=False, indent=2))
